In [1]:
#حالا میخوام پارامترهای الگوریتم رندوم فورست رو تغییر بدم تا ببینم میتونم مدلم رو بهبود بدم و امتیاز کراس ولیدیشن بهتری بگیرم
#به این کار میگیم Hyperparameter Tuning 

#برای RandomForestRegressor ده‌ها پارامتر وجود دارد، اما این‌ها مهم‌ترین‌ها هستند
#| پارامتر             | معنی خیلی ساده                                             |
#| ------------------- | --------------------------------------------------------|

#| `n_estimators`      | چند تا درخت ساخته شود؟|RandomForestRegressor(n_estimators=100) ینی ۱۰۰ درخت تصمیم بساز و در آخر میانگین جوابشان را بگیر.

#| `max_depth`         | هر درخت حداکثر چقدر عمیق شود؟ |RandomForestRegressor(max_depth=5) ینی هر درخت فقط اجازه دارد ۵ سطح سؤال بپرسد. اگر خیلی بزرگ باشد:درخت ممکن است داده را حفظ کند (Overfitting).اگر خیلی کوچک باشد:درخت چیز زیادی یاد نمی‌گیرد (Underfitting).

#| `min_samples_split` | یک گره برای تقسیم شدن حداقل چند نمونه داشته باشد؟|

#| `min_samples_leaf`  | هر برگ حداقل چند نمونه داشته باشد؟ |RandomForestRegressor(min_samples_leaf=10) ینی هیچ برگی نباید کمتر از ۱۰ خانه داشته باشد.این باعث می‌شود مدل کمتر روی داده‌های عجیب حساس شود.

#| `max_features`      | هر بار برای تقسیم، چند feature بررسی شوند؟ |   
#فرض کن 35 تا ویژگی  داری
#max_features=10 ینی هر بار که درخت می‌خواهد سؤال بعدی را انتخاب کند، فقط از بین ۱۰ ویژگی تصادفی انتخاب کند، نه هر ۳۵ تا. 
#این یکی از دلایل قدرت Random Forest است؛ باعث می‌شود درخت‌ها شبیه هم نباشند.


#| `bootstrap`         | هر درخت از کل داده استفاده کند یا از یک نمونه‌گیری تصادفی؟ |
#| `random_state`      | برای ثابت نگه داشتن نتیجه‌های تصادفی |



import pandas as pd
df = pd.read_csv("train.csv")

#پرکردن خانه های خالی از جنس استرینگ
df["PoolQC"] = df["PoolQC"].fillna("No Pool")
df["MiscFeature"] = df["MiscFeature"].fillna("None")
df["Alley"] = df["Alley"].fillna("No alley access")
df["Fence"] = df["Fence"].fillna("No Fence")
df["MasVnrType"] = df["MasVnrType"].fillna("None")
df["FireplaceQu"] = df["FireplaceQu"].fillna("No Fireplace")
df["GarageQual"] = df["GarageQual"].fillna("No Garage")
df["GarageFinish"] = df["GarageFinish"].fillna("No Garage")
df["GarageType"] = df["GarageType"].fillna("No Garage")
df["GarageCond"] = df["GarageCond"].fillna("No Garage")
df["BsmtFinType2"] = df["BsmtFinType2"].fillna("No Basement")
df["BsmtExposure"] = df["BsmtExposure"].fillna("No Basement")
df["BsmtCond"] = df["BsmtCond"].fillna("No Basement")
df["BsmtQual"] = df["BsmtQual"].fillna("No Basement")
df["BsmtFinType1"] = df["BsmtFinType1"].fillna("No Basement")
df["Electrical"] = df["Electrical"].fillna(df["Electrical"].mode()[0])

#پرکردن خانه های خالی از جنس عدد
df["LotFrontage"] = df["LotFrontage"].fillna(df["LotFrontage"].median())

df["GarageYrBlt"] = df["GarageYrBlt"].fillna(0)

df["MasVnrArea"] = df["MasVnrArea"].fillna(0)

#تعیین x,y
X = df.drop(["Id", "SalePrice"], axis=1)
y = df["SalePrice"]

#تبدیل داده استرینگ به عدد
X = pd.get_dummies(X)

#تقسیم دیتا به دو بخش آموزش و تست
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#انتخاب مدل
from sklearn.ensemble import RandomForestRegressor

###################################################################################
#از اینجا جدیده

# حالا می‌خواهیم روی الگوریتم رندوم فورست یک پارامتر را تغییر بدیم و اثرش را ببینیم
#مثلا پارامتر  max_depth  مقدارشو کم کنیم رو تغییر میدم
# میخوایم عمق پرسیدن سوال رو محدود کنیم ینی درخت ها بیشتر از 10 مرحله سوال نپرسن

model = RandomForestRegressor(max_depth=10, random_state=42)

#آموزش و تست
model.fit(X_train, y_train)
model.score(X_test, y_test)


0.8881810947242009

In [2]:
#میانگین Score = عملکرد واقعی‌تر مدل
from sklearn.model_selection import cross_val_score
scores = cross_val_score(model,X,y,cv=5)
scores.mean()

np.float64(0.8547324612975608)

In [6]:
#حالا میخوایم یه پارامتر دیگه رو تغییر بدیم ایندفه n_estimators رو
#مثلا مقدار پیش فرض برای تعداد درخت ها رو بیشتر کنیم درخت بیشتر → معمولاً مدل پایدارتر می‌شود (البته کندتر)
#ینی n_estimators رو از دیفالتش که 100 هست به 300 تغییر بدیم مقدارشو زیاد کنیم 

model = RandomForestRegressor(
    n_estimators=300, #ینی Random Forest چند تا درخت بسازد؟
    random_state=42 #ینی تصادفی بودن مدل را ثابت کن تا هر بار نتیجه متفاوت نشود
)

#آموزش و تست
model.fit(X_train, y_train)
model.score(X_test, y_test)

0.8921573592467097

In [4]:
scores = cross_val_score(model,X,y,cv=5)
scores.mean()

np.float64(0.8558891289255339)

In [5]:
#نتیجه تا به اینجا
#اگر فقط به Cross Validation نگاه کنیم
#هنوز همان مدل پیش‌فرض ینی قبل تغییر هیچ پارامتری که در انتهای تمرین سوم بهش رسیدیم، بهترین انتخاب است
#دلیلش هم این است که معیار قابل اعتمادتر، میانگین Cross Validation است،
#نه فقط یک score روی یک تقسیم داده.